# Bibliotecas

In [16]:
import pandas as pd 
import matplotlib.pyplot as plt 
import numpy as np 
import statistics
import seaborn as sns
from scipy.stats import levene
from scipy.stats import shapiro
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import kruskal
import scikit_posthocs as sp
from scipy.stats import levene, shapiro, kruskal
import pingouin as pg

# Funções

In [17]:
# Funções auxiliares
def run_levene(df, feature, group_col='concentration'):
    groups = [df.loc[df[group_col] == c, feature] for c in df[group_col].unique()]
    stat, p = levene(*groups)
    status = "homocedasticidade" if p > 0.05 else "heterocedasticidade"
    print(f"Levene p-valor ({feature}) = {p:.4f} -> {status}")
    return p > 0.05  # True se homocedástico


def run_shapiro(df, feature, group_col='concentration'):
    normal_all = True
    for c in sorted(df[group_col].unique()):
        data = df.loc[df[group_col] == c, feature]
        stat, p = shapiro(data)
        normality = "distribuição normal" if p > 0.05 else "não normal"
        print(f"Concentração {c}: p = {p:.4f} -> {normality}")
        if p <= 0.05:
            normal_all = False
    return normal_all  # True se todos os grupos normais


def run_anova_oneway(df, feature, group_col='concentration', normal=True, homoced=True):
    if normal and homoced:
        # ANOVA clássico
        print("\n=== ANOVA One-way (clássico) ===")
        model = ols(f'{feature} ~ C({group_col})', data=df).fit()
        anova_results = anova_lm(model)
        print(anova_results)
        return "anova"

    elif normal and not homoced:
        # Welch ANOVA
        print("\n=== Welch ANOVA (heterocedasticidade) ===")
        welch = pg.welch_anova(dv=feature, between=group_col, data=df)
        print(welch)
        return "welch"

    else:
        # Kruskal-Wallis
        print("\n=== Kruskal-Wallis (não paramétrico) ===")
        groups = [df.loc[df[group_col] == c, feature] for c in df[group_col].unique()]
        stat, p = kruskal(*groups)
        print(f"estatística = {stat:.4f}, p = {p:.4f}")
        return "kruskal"


def run_posthoc(df, feature, group_col='concentration', method="anova"):
    if method == "anova":
        print("\n--- Tukey HSD ---")
        tukey = pairwise_tukeyhsd(endog=df[feature],
                                  groups=df[group_col],
                                  alpha=0.05)
        print(tukey)

    elif method == "welch":
        print("\n--- Games-Howell ---")
        gh = pg.pairwise_gameshowell(dv=feature, between=group_col, data=df)
        print(gh)

    elif method == "kruskal":
        print("\n--- Dunn test (pós-hoc não paramétrico) ---")
        dunn = sp.posthoc_dunn(df, val_col=feature, group_col=group_col, p_adjust='bonferroni')
        print(dunn)


def run_anova_twoway(df, feature, factor1='concentration', factor2='compound'):
    print("\n=== Two-way ANOVA (clássico, assumir normalidade/homocedasticidade) ===")
    model = ols(f'{feature} ~ C({factor1}) + C({factor2}) + C({factor1}):C({factor2})', data=df).fit()
    anova_results = anova_lm(model)
    print(anova_results)

    for factor in [f"C({factor1})", f"C({factor2})", f"C({factor1}):C({factor2})"]:
        p_value = anova_results.loc[factor, 'PR(>F)']
        if p_value < 0.05:
            print(f"{factor}: efeito significativo (p = {p_value:.4f})")
        else:
            print(f"{factor}: sem efeito significativo (p = {p_value:.4f})")


# Dataset

In [18]:
df = pd.read_excel("../../bancos de dados/dataset_premium.xlsx")

controle = df[df["type"] == "Control"]

controle_BCT = controle.copy()
controle_BCT["type"] = "BCT"

controle_BST = controle.copy()
controle_BST["type"] = "BST"

df_exp = pd.concat([df, controle_BCT, controle_BST], ignore_index=True)

df_exp = df_exp[df_exp['type'] != 'Control']

df = df[0:60]

display(df)

,sample,type,concentration,bpm,body_length,yolk_sac_area,pigmented_area,open_swim_bladder,edema_present,deformation_present,aquisition_data
0,1_bct_1,BCT,1.0,115,3.721,0.396,18096,no,no,no,2025-08-18T15:35:09.206382-03:00
1,2_bct_1,BCT,1.0,133,3.933,0.432,19992,no,no,no,2025-08-18T15:41:30.4801149-03:00
2,3_bct_1,BCT,1.0,149,3.690,0.245,23613,no,no,yes,2025-08-18T15:45:05.6896737-03:00
3,4_bct_1,BCT,1.0,124,3.909,0.362,27886,no,no,no,2025-08-18T15:49:14.9293665-03:00
4,5_bct_1,BCT,1.0,124,4.302,0.501,27014,no,no,no,2025-08-18T15:56:33.7860505-03:00
5,6_bct_1,BCT,1.0,129,3.414,0.396,15959,no,yes,yes,2025-08-18T16:02:00.6964589-03:00
6,7_bct_1,BCT,1.0,148,3.707,0.349,24064,no,no,no,2025-08-18T16:08:26.0680252-03:00
7,8_bct_1,BCT,1.0,133,3.915,0.355,20781,no,yes,no,2025-08-18T16:15:37.3798923-03:00
8,9_bct_1,BCT,1.0,137,3.857,0.309,18757,no,no,yes,2025-08-18T16:22:04.9105528-03:00
9,1_bst_1,BST,1.0,119,3.901,0.239,26009,no,no,yes,2025-08-18T16:30:54.3457357-03:00


# Anova

Três regras precisam ser satisfeitas:
- Independência das observações (Cada peixe é um indivíduo e não interfere nos resultados de outros)
- Homogenidade na variância
- Distribuição Normal

In [19]:
df = pd.read_excel("../../bancos de dados/dataset_premium.xlsx")

df_BCT = df[df["type"] != "BST"]
df_BST = df[df["type"] != "BCT"]

df_BCT = df_BCT.copy()
df_BST = df_BST.copy()

bct_c = []

for i in range(len(df_BCT)):
    bct_c.append('BCT')

bst_c = []

for i in range(len(df_BST)):
    bst_c.append('BST')

df_BCT['type'] = bct_c
df_BST['type'] = bst_c

display(df)

,sample,type,concentration,bpm,body_length,yolk_sac_area,pigmented_area,open_swim_bladder,edema_present,deformation_present,aquisition_data
0,1_bct_1,BCT,1.0,115,3.721,0.396,18096,no,no,no,2025-08-18T15:35:09.206382-03:00
1,2_bct_1,BCT,1.0,133,3.933,0.432,19992,no,no,no,2025-08-18T15:41:30.4801149-03:00
2,3_bct_1,BCT,1.0,149,3.690,0.245,23613,no,no,yes,2025-08-18T15:45:05.6896737-03:00
3,4_bct_1,BCT,1.0,124,3.909,0.362,27886,no,no,no,2025-08-18T15:49:14.9293665-03:00
4,5_bct_1,BCT,1.0,124,4.302,0.501,27014,no,no,no,2025-08-18T15:56:33.7860505-03:00
...,...,...,...,...,...,...,...,...,...,...,...
90,8_control,Control,0.0,152,3.849,0.266,11241,no,no,no,NaN
91,9_control,Control,0.0,172,3.971,0.326,13793,yes,no,no,NaN
92,10_control,Control,0.0,168,3.966,0.262,11152,no,no,no,NaN
93,11_control,Control,0.0,152,3.974,0.305,12402,yes,no,yes,NaN


# Parâmetros Premium

## BPM

In [20]:
features = ['bpm']
compounds = {'BCT': df_BCT, 'BST': df_BST}

# rodar testes individuais por composto
for compound_name, df in compounds.items():
    print(f"\n=== {compound_name} ===")
    for feature in features:
        print(f"\nVariável: {feature}")
        print()

        # testes de pré-condição
        homoced = run_levene(df, feature)
        normal = run_shapiro(df, feature)

        # ANOVA adequada
        method = run_anova_oneway(df, feature, normal=normal, homoced=homoced)

        # Pós-teste adequado
        run_posthoc(df, feature, method=method)


# rodar two-way ANOVA com todos os compostos juntos
df_all = pd.concat([df_BCT.assign(compound='BCT'),
                    df_BST.assign(compound='BST')],
                   ignore_index=True)

for feature in features:
    print(f"\n=== ANOVA Two-way: {feature} ===")
    run_anova_twoway(df_all, feature)



=== BCT ===

Variável: bpm

Levene p-valor (bpm) = 0.6004 -> homocedasticidade
Concentração 0.0: p = 0.2125 -> distribuição normal
Concentração 0.5: p = 0.7464 -> distribuição normal
Concentração 1.0: p = 0.6834 -> distribuição normal
Concentração 2.0: p = 0.8795 -> distribuição normal

=== ANOVA One-way (clássico) ===
                    df       sum_sq      mean_sq          F        PR(>F)
C(concentration)   3.0  9827.134746  3275.711582  26.090981  6.317112e-11
Residual          60.0  7532.974629   125.549577        NaN           NaN

--- Tukey HSD ---
 Multiple Comparison of Means - Tukey HSD, FWER=0.05  
group1 group2 meandiff p-adj   lower    upper   reject
------------------------------------------------------
   0.0    0.5 -21.3021    0.0 -31.5728 -11.0315   True
   0.0    1.0  -21.585    0.0 -32.6844 -10.4856   True
   0.0    2.0 -29.8294    0.0  -40.481 -19.1779   True
   0.5    1.0  -0.2828 0.9999 -13.5912  13.0255  False
   0.5    2.0  -8.5273 0.3116 -21.4644   4.4099  Fal

## Comprimento

In [ ]:
features = ['body_length']
compounds = {'BCT': df_BCT, 'BST': df_BST}

display(df_BCT)

# rodar testes individuais por composto
for compound_name, df in compounds.items():
    print(f"\n=== {compound_name} ===")
    for feature in features:
        print(f"\nVariável: {feature}")
        print()

        # testes de pré-condição
        homoced = run_levene(df, feature)
        normal = run_shapiro(df, feature)

        # ANOVA adequada
        method = run_anova_oneway(df, feature, normal=normal, homoced=homoced)

        # Pós-teste adequado
        run_posthoc(df, feature, method=method)


# rodar two-way ANOVA com todos os compostos juntos
df_all = pd.concat([df_BCT.assign(compound='BCT'),
                    df_BST.assign(compound='BST')],
                   ignore_index=True)

for feature in features:
    print(f"\n=== ANOVA Two-way: {feature} ===")
    run_anova_twoway(df_all, feature)


,sample,type,concentration,bpm,body_length,yolk_sac_area,pigmented_area,open_swim_bladder,edema_present,deformation_present,aquisition_data
0,1_bct_1,BCT,1.0,115,3.721,0.396,18096,no,no,no,2025-08-18T15:35:09.206382-03:00
1,2_bct_1,BCT,1.0,133,3.933,0.432,19992,no,no,no,2025-08-18T15:41:30.4801149-03:00
2,3_bct_1,BCT,1.0,149,3.690,0.245,23613,no,no,yes,2025-08-18T15:45:05.6896737-03:00
3,4_bct_1,BCT,1.0,124,3.909,0.362,27886,no,no,no,2025-08-18T15:49:14.9293665-03:00
4,5_bct_1,BCT,1.0,124,4.302,0.501,27014,no,no,no,2025-08-18T15:56:33.7860505-03:00
...,...,...,...,...,...,...,...,...,...,...,...
90,8_control,BCT,0.0,152,3.849,0.266,11241,no,no,no,NaN
91,9_control,BCT,0.0,172,3.971,0.326,13793,yes,no,no,NaN
92,10_control,BCT,0.0,168,3.966,0.262,11152,no,no,no,NaN
93,11_control,BCT,0.0,152,3.974,0.305,12402,yes,no,yes,NaN



=== BCT ===

Variável: body_length

Levene p-valor (body_length) = 0.5629 -> homocedasticidade
Concentração 0.0: p = 0.0000 -> não normal
Concentração 0.5: p = 0.5278 -> distribuição normal
Concentração 1.0: p = 0.5219 -> distribuição normal
Concentração 2.0: p = 0.2586 -> distribuição normal

=== Kruskal-Wallis (não paramétrico) ===
estatística = 28.0998, p = 0.0000

--- Dunn test (pós-hoc não paramétrico) ---
          0.0       0.5       1.0       2.0
0.0  1.000000  0.016981  1.000000  0.000004
0.5  0.016981  1.000000  1.000000  0.523865
1.0  1.000000  1.000000  1.000000  0.023123
2.0  0.000004  0.523865  0.023123  1.000000

=== BST ===

Variável: body_length

Levene p-valor (body_length) = 0.5849 -> homocedasticidade
Concentração 0.0: p = 0.0000 -> não normal
Concentração 0.5: p = 0.9007 -> distribuição normal
Concentração 1.0: p = 0.0283 -> não normal
Concentração 2.0: p = 0.7326 -> distribuição normal

=== Kruskal-Wallis (não paramétrico) ===
estatística = 21.2091, p = 0.0001

-

## Área do Saco

In [22]:
features = ['yolk_sac_area']
compounds = {'BCT': df_BCT, 'BST': df_BST}

# rodar testes individuais por composto
for compound_name, df in compounds.items():
    print(f"\n=== {compound_name} ===")
    for feature in features:
        print(f"\nVariável: {feature}")
        print()

        # testes de pré-condição
        homoced = run_levene(df, feature)
        normal = run_shapiro(df, feature)

        # ANOVA adequada
        method = run_anova_oneway(df, feature, normal=normal, homoced=homoced)

        # Pós-teste adequado
        run_posthoc(df, feature, method=method)

# rodar two-way ANOVA com todos os compostos juntos
df_all = pd.concat([df_BCT.assign(compound='BCT'),
                    df_BST.assign(compound='BST')],
                   ignore_index=True)

for feature in features:
    print(f"\n=== ANOVA Two-way: {feature} ===")
    run_anova_twoway(df_all, feature)



=== BCT ===

Variável: yolk_sac_area

Levene p-valor (yolk_sac_area) = 0.0569 -> homocedasticidade
Concentração 0.0: p = 0.9233 -> distribuição normal
Concentração 0.5: p = 0.0802 -> distribuição normal
Concentração 1.0: p = 0.9610 -> distribuição normal
Concentração 2.0: p = 0.3571 -> distribuição normal

=== ANOVA One-way (clássico) ===
                    df    sum_sq   mean_sq          F        PR(>F)
C(concentration)   3.0  0.108389  0.036130  19.997012  4.195404e-09
Residual          60.0  0.108405  0.001807        NaN           NaN

--- Tukey HSD ---
Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj   lower  upper  reject
---------------------------------------------------
   0.0    0.5   0.0599 0.0008  0.0209 0.0988   True
   0.0    1.0   0.1073    0.0  0.0651 0.1494   True
   0.0    2.0   0.0712 0.0001  0.0308 0.1116   True
   0.5    1.0   0.0474 0.0733 -0.0031 0.0979  False
   0.5    2.0   0.0113 0.9285 -0.0378 0.0604  False
   1.0    2.0  -0.0

## Área Pigmentada

In [23]:
features = ['pigmented_area']
compounds = {'BCT': df_BCT, 'BST': df_BST}

# rodar testes individuais por composto
for compound_name, df in compounds.items():
    print(f"\n=== {compound_name} ===")
    for feature in features:
        print(f"\nVariável: {feature}")
        print()

        # testes de pré-condição
        homoced = run_levene(df, feature)
        normal = run_shapiro(df, feature)

        # ANOVA adequada
        method = run_anova_oneway(df, feature, normal=normal, homoced=homoced)

        # Pós-teste adequado
        run_posthoc(df, feature, method=method)

# rodar two-way ANOVA com todos os compostos juntos
df_all = pd.concat([df_BCT.assign(compound='BCT'),
                    df_BST.assign(compound='BST')],
                   ignore_index=True)

for feature in features:
    print(f"\n=== ANOVA Two-way: {feature} ===")
    run_anova_twoway(df_all, feature)



=== BCT ===

Variável: pigmented_area

Levene p-valor (pigmented_area) = 0.0125 -> heterocedasticidade
Concentração 0.0: p = 0.0313 -> não normal
Concentração 0.5: p = 0.2177 -> distribuição normal
Concentração 1.0: p = 0.7354 -> distribuição normal
Concentração 2.0: p = 0.7891 -> distribuição normal

=== Kruskal-Wallis (não paramétrico) ===
estatística = 45.5116, p = 0.0000

--- Dunn test (pós-hoc não paramétrico) ---
          0.0       0.5      1.0       2.0
0.0  1.000000  0.000068  0.00001  0.000006
0.5  0.000068  1.000000  1.00000  1.000000
1.0  0.000010  1.000000  1.00000  1.000000
2.0  0.000006  1.000000  1.00000  1.000000

=== BST ===

Variável: pigmented_area

Levene p-valor (pigmented_area) = 0.1161 -> homocedasticidade
Concentração 0.0: p = 0.0313 -> não normal
Concentração 0.5: p = 0.2475 -> distribuição normal
Concentração 1.0: p = 0.4088 -> distribuição normal
Concentração 2.0: p = 0.4879 -> distribuição normal

=== Kruskal-Wallis (não paramétrico) ===
estatística = 44.1

# Cometa

In [24]:
df_cometa = pd.read_csv("../../bancos de dados/dataset_cometa.csv", sep=';')

df_cometa = df_cometa.rename(columns={'tail moment': 'tail_moment'})

display(df_cometa)

df_cometa['tail_moment'] = (
    df_cometa['tail_moment']
    .astype(str)                 # garante que tudo seja string
    .str.replace(',', '.', regex=False)  # troca vírgula por ponto
    .str.strip()                 # remove espaços
    .replace(['', ' ', 'nan', 'NaN', None], pd.NA)  # limpa valores inválidos
    .astype(float)               # converte para float
)

df_cometa['concentration'] = (
    df_cometa['concentration']
    .astype(str)                 # garante que tudo seja string
    .str.replace(',', '.', regex=False)  # troca vírgula por ponto
    .str.strip()                 # remove espaços
    .replace(['', ' ', 'nan', 'NaN', None], pd.NA)  # limpa valores inválidos
    .astype(float)               # converte para float
)

df_cometa.to_excel('cometa_limpo.xlsx', index=False)

df_cometa = df_cometa[df_cometa['concentration'] != 'Positive']

df_cometa



,type,concentration,tail_moment
0,control_negative,0.0,"0,97"
1,control_negative,0.0,"1,71"
2,control_negative,0.0,"1,08"
3,control_negative,0.0,"1,14"
4,control_negative,0.0,"1,63"
...,...,...,...
862,control_positive,0.0,"3,45"
863,control_positive,0.0,"22,24"
864,control_positive,0.0,"2,38"
865,control_positive,0.0,"44,49"


,type,concentration,tail_moment
0,control_negative,0.0,0.97
1,control_negative,0.0,1.71
2,control_negative,0.0,1.08
3,control_negative,0.0,1.14
4,control_negative,0.0,1.63
...,...,...,...
862,control_positive,0.0,3.45
863,control_positive,0.0,22.24
864,control_positive,0.0,2.38
865,control_positive,0.0,44.49


In [25]:
df_BCT = df_cometa[df_cometa["type"] != "BST"]
df_BST = df_cometa[df_cometa["type"] != "BCT"]

df_BCT = df_BCT.copy()
df_BST = df_BST.copy()

bct_c = []

for i in range(len(df_BCT)):
    bct_c.append('BCT')

bst_c = []

for i in range(len(df_BST)):
    bst_c.append('BST')

df_BCT['type'] = bct_c
df_BST['type'] = bst_c

display(df_BCT)

,type,concentration,tail_moment
0,BCT,0.0,0.97
1,BCT,0.0,1.71
2,BCT,0.0,1.08
3,BCT,0.0,1.14
4,BCT,0.0,1.63
...,...,...,...
862,BCT,0.0,3.45
863,BCT,0.0,22.24
864,BCT,0.0,2.38
865,BCT,0.0,44.49


In [26]:
features = ['tail_moment']
compounds = {'BCT': df_BCT, 'BST': df_BST}

# rodar testes individuais por composto
for compound_name, df in compounds.items():
    print(f"\n=== {compound_name} ===")
    for feature in features:
        print(f"\nVariável: {feature}")
        print()

        # testes de pré-condição
        homoced = run_levene(df, feature)
        normal = run_shapiro(df, feature)

        # ANOVA adequada
        method = run_anova_oneway(df, feature, normal=normal, homoced=homoced)

        # Pós-teste adequado
        run_posthoc(df, feature, method=method)

# rodar two-way ANOVA com todos os compostos juntos
df_all = pd.concat([df_BCT.assign(compound='BCT'),
                    df_BST.assign(compound='BST')],
                   ignore_index=True)

for feature in features:
    print(f"\n=== ANOVA Two-way: {feature} ===")
    run_anova_twoway(df_all, feature)



=== BCT ===

Variável: tail_moment

Levene p-valor (tail_moment) = 0.0004 -> heterocedasticidade
Concentração 0.0: p = 0.0000 -> não normal
Concentração 0.5: p = 0.0000 -> não normal
Concentração 1.0: p = 0.0000 -> não normal
Concentração 2.0: p = 0.0000 -> não normal

=== Kruskal-Wallis (não paramétrico) ===
estatística = 62.8095, p = 0.0000

--- Dunn test (pós-hoc não paramétrico) ---
              0.0           0.5           1.0       2.0
0.0  1.000000e+00  3.428687e-10  5.024326e-08  0.000037
0.5  3.428687e-10  1.000000e+00  1.000000e+00  0.174966
1.0  5.024326e-08  1.000000e+00  1.000000e+00  0.299206
2.0  3.720916e-05  1.749660e-01  2.992063e-01  1.000000

=== BST ===

Variável: tail_moment

Levene p-valor (tail_moment) = 0.0000 -> heterocedasticidade
Concentração 0.0: p = 0.0000 -> não normal
Concentração 0.5: p = 0.0000 -> não normal
Concentração 1.0: p = 0.0000 -> não normal
Concentração 2.0: p = 0.0000 -> não normal

=== Kruskal-Wallis (não paramétrico) ===
estatística = 104

# RTqPCR

In [27]:
df = pd.read_excel("../../bancos de dados/dataset_rtqpcr.xlsx")

df_BCT = df[df["type"] != "bst"]
df_BST = df[df["type"] != "bct"]

df_BCT = df_BCT.copy()
df_BST = df_BST.copy()

bct_c = []

for i in range(len(df_BCT)):
    bct_c.append('bct')

bst_c = []

for i in range(len(df_BST)):
    bst_c.append('bst')

df_BCT['type'] = bct_c
df_BST['type'] = bst_c

display(df_BCT)

,type,concentration,nestin,gfap,il1,tnf,casp9
0,bct,0.0,0.94400,0.9550,0.9470,0.9691,0.9254
1,bct,0.0,1.00000,0.9741,1.0540,1.0318,1.0140
2,bct,0.0,1.05400,1.0740,NaN,NaN,1.0640
3,bct,0.5,0.95380,0.9920,0.4823,0.6748,1.6640
4,bct,0.5,0.75019,1.0190,0.6556,0.7795,1.5730
5,bct,0.5,0.89670,0.9890,0.5304,0.7670,1.7420


In [28]:
features = ['nestin']
compounds = {'BCT': df_BCT, 'BST': df_BST}

# rodar testes individuais por composto
for compound_name, df in compounds.items():
    print(f"\n=== {compound_name} ===")
    for feature in features:
        display(df)
        print(f"\nVariável: {feature}")
        print()

        # testes de pré-condição
        homoced = run_levene(df, feature)
        normal = run_shapiro(df, feature)

        # ANOVA adequada
        method = run_anova_oneway(df, feature, normal=normal, homoced=homoced)

        # Pós-teste adequado
        run_posthoc(df, feature, method=method)

# rodar two-way ANOVA com todos os compostos juntos
df_all = pd.concat([df_BCT.assign(compound='BCT'),
                    df_BST.assign(compound='BST')],
                   ignore_index=True)

for feature in features:
    print(f"\n=== ANOVA Two-way: {feature} ===")
    run_anova_twoway(df_all, feature)



=== BCT ===


,type,concentration,nestin,gfap,il1,tnf,casp9
0,bct,0.0,0.94400,0.9550,0.9470,0.9691,0.9254
1,bct,0.0,1.00000,0.9741,1.0540,1.0318,1.0140
2,bct,0.0,1.05400,1.0740,NaN,NaN,1.0640
3,bct,0.5,0.95380,0.9920,0.4823,0.6748,1.6640
4,bct,0.5,0.75019,1.0190,0.6556,0.7795,1.5730
5,bct,0.5,0.89670,0.9890,0.5304,0.7670,1.7420



Variável: nestin

Levene p-valor (nestin) = 0.5382 -> homocedasticidade
Concentração 0.0: p = 0.9800 -> distribuição normal
Concentração 0.5: p = 0.5258 -> distribuição normal

=== ANOVA One-way (clássico) ===
                   df    sum_sq   mean_sq         F    PR(>F)
C(concentration)  1.0  0.026309  0.026309  3.743545  0.125119
Residual          4.0  0.028112  0.007028       NaN       NaN

--- Tukey HSD ---
Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj   lower  upper  reject
---------------------------------------------------
   0.0    0.5  -0.1324 0.1251 -0.3225 0.0576  False
---------------------------------------------------

=== BST ===


,type,concentration,nestin,gfap,il1,tnf,casp9
0,bst,0.0,0.9440,0.9550,0.9470,0.969100,0.9254
1,bst,0.0,1.0000,0.9741,1.0540,1.031800,1.0140
2,bst,0.0,1.0540,1.0740,NaN,NaN,1.0640
6,bst,0.5,0.8483,0.9530,0.5954,0.362714,1.4540
7,bst,0.5,0.6586,0.9940,0.4400,0.350981,1.6650
8,bst,0.5,NaN,1.0940,NaN,0.374276,NaN



Variável: nestin

Levene p-valor (nestin) = nan -> heterocedasticidade
Concentração 0.0: p = 0.9800 -> distribuição normal
Concentração 0.5: p = nan -> não normal

=== Welch ANOVA (heterocedasticidade) ===
          Source  ddof1     ddof2        F     p-unc       np2
0  concentration      1  1.229029  6.04286  0.206869  0.751085

--- Games-Howell ---
     A    B   mean(A)  mean(B)      diff        se         T        df  \
0  0.0  0.5  0.999333  0.75345  0.245883  0.100025  2.458223  1.229029   

       pval    hedges  
0  0.206869  1.997498  

=== ANOVA Two-way: nestin ===

=== Two-way ANOVA (clássico, assumir normalidade/homocedasticidade) ===
                               df    sum_sq   mean_sq          F    PR(>F)
C(concentration)              1.0  0.086232  0.086232  11.573560  0.011411
C(compound)                   1.0  0.006864  0.006864   0.921260  0.369116
C(concentration):C(compound)  1.0  0.008580  0.008580   1.151575  0.318814
Residual                      7.0  0.052155 

In [29]:
features['gfap']

compounds = {'BCT': df_BCT, 'BST': df_BST}

# rodar testes individuais por composto
for compound_name, df in compounds.items():
    print(f"\n=== {compound_name} ===")
    for feature in features:
        print(f"\nVariável: {feature}")
        print()

        # testes de pré-condição
        homoced = run_levene(df, feature)
        normal = run_shapiro(df, feature)

        # ANOVA adequada
        method = run_anova_oneway(df, feature, normal=normal, homoced=homoced)

        # Pós-teste adequado
        run_posthoc(df, feature, method=method)

# rodar two-way ANOVA com todos os compostos juntos
df_all = pd.concat([df_BCT.assign(compound='BCT'),
                    df_BST.assign(compound='BST')],
                   ignore_index=True)

for feature in features:
    print(f"\n=== ANOVA Two-way: {feature} ===")
    run_anova_twoway(df_all, feature)


TypeError: list indices must be integers or slices, not str

In [30]:
features = ['il1']

compounds = {'BCT': df_BCT, 'BST': df_BST}

# rodar testes individuais por composto
for compound_name, df in compounds.items():
    print(f"\n=== {compound_name} ===")
    for feature in features:
        print(f"\nVariável: {feature}")
        print()

        # testes de pré-condição
        homoced = run_levene(df, feature)
        normal = run_shapiro(df, feature)

        # ANOVA adequada
        method = run_anova_oneway(df, feature, normal=normal, homoced=homoced)

        # Pós-teste adequado
        run_posthoc(df, feature, method=method)

# rodar two-way ANOVA com todos os compostos juntos
df_all = pd.concat([df_BCT.assign(compound='BCT'),
                    df_BST.assign(compound='BST')],
                   ignore_index=True)

for feature in features:
    print(f"\n=== ANOVA Two-way: {feature} ===")
    run_anova_twoway(df_all, feature)



=== BCT ===

Variável: il1

Levene p-valor (il1) = nan -> heterocedasticidade
Concentração 0.0: p = nan -> não normal
Concentração 0.5: p = 0.5198 -> distribuição normal

=== Welch ANOVA (heterocedasticidade) ===
          Source  ddof1     ddof2          F     p-unc       np2
0  concentration      1  2.602466  35.711991  0.013873  0.916004

--- Games-Howell ---
     A    B  mean(A)  mean(B)    diff        se         T        df      pval  \
0  0.0  0.5   1.0005   0.5561  0.4444  0.074365  5.975951  2.602466  0.013873   

     hedges  
0  3.797391  

=== BST ===

Variável: il1

Levene p-valor (il1) = nan -> heterocedasticidade
Concentração 0.0: p = nan -> não normal
Concentração 0.5: p = nan -> não normal

=== Welch ANOVA (heterocedasticidade) ===
          Source  ddof1     ddof2          F     p-unc       np2
0  concentration      1  1.774181  26.191898  0.046024  0.929058

--- Games-Howell ---
     A    B  mean(A)  mean(B)    diff        se         T        df      pval  \
0  0.0  

In [31]:
features = ['tnf']

compounds = {'BCT': df_BCT, 'BST': df_BST}

# rodar testes individuais por composto
for compound_name, df in compounds.items():
    print(f"\n=== {compound_name} ===")
    for feature in features:
        print(f"\nVariável: {feature}")
        print()

        # testes de pré-condição
        homoced = run_levene(df, feature)
        normal = run_shapiro(df, feature)

        # ANOVA adequada
        method = run_anova_oneway(df, feature, normal=normal, homoced=homoced)

        # Pós-teste adequado
        run_posthoc(df, feature, method=method)

# rodar two-way ANOVA com todos os compostos juntos
df_all = pd.concat([df_BCT.assign(compound='BCT'),
                    df_BST.assign(compound='BST')],
                   ignore_index=True)

for feature in features:
    print(f"\n=== ANOVA Two-way: {feature} ===")
    run_anova_twoway(df_all, feature)



=== BCT ===

Variável: tnf

Levene p-valor (tnf) = nan -> heterocedasticidade
Concentração 0.0: p = nan -> não normal
Concentração 0.5: p = 0.2092 -> distribuição normal

=== Welch ANOVA (heterocedasticidade) ===
          Source  ddof1    ddof2          F     p-unc       np2
0  concentration      1  2.75421  32.617435  0.013386  0.905112

--- Games-Howell ---
     A    B  mean(A)   mean(B)      diff        se         T       df  \
0  0.0  0.5  1.00045  0.740433  0.260017  0.045528  5.711168  2.75421   

       pval    hedges  
0  0.013386  3.551505  

=== BST ===

Variável: tnf

Levene p-valor (tnf) = nan -> heterocedasticidade
Concentração 0.0: p = nan -> não normal
Concentração 0.5: p = 0.9919 -> distribuição normal

=== Welch ANOVA (heterocedasticidade) ===
          Source  ddof1     ddof2           F     p-unc       np2
0  concentration      1  1.092985  395.683085  0.024719  0.995438

--- Games-Howell ---
     A    B  mean(A)   mean(B)      diff        se          T        df  

In [32]:
features = ['casp9']

compounds = {'BCT': df_BCT, 'BST': df_BST}

# rodar testes individuais por composto
for compound_name, df in compounds.items():
    print(f"\n=== {compound_name} ===")
    for feature in features:
        print(f"\nVariável: {feature}")
        print()

        # testes de pré-condição
        homoced = run_levene(df, feature)
        normal = run_shapiro(df, feature)

        # ANOVA adequada
        method = run_anova_oneway(df, feature, normal=normal, homoced=homoced)

        # Pós-teste adequado
        run_posthoc(df, feature, method=method)

# rodar two-way ANOVA com todos os compostos juntos
df_all = pd.concat([df_BCT.assign(compound='BCT'),
                    df_BST.assign(compound='BST')],
                   ignore_index=True)

for feature in features:
    print(f"\n=== ANOVA Two-way: {feature} ===")
    run_anova_twoway(df_all, feature)



=== BCT ===

Variável: casp9

Levene p-valor (casp9) = 0.8043 -> homocedasticidade
Concentração 0.0: p = 0.6955 -> distribuição normal
Concentração 0.5: p = 0.9152 -> distribuição normal

=== ANOVA One-way (clássico) ===
                   df    sum_sq   mean_sq           F    PR(>F)
C(concentration)  1.0  0.650499  0.650499  107.689752  0.000487
Residual          4.0  0.024162  0.006040         NaN       NaN

--- Tukey HSD ---
Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj  lower  upper  reject
--------------------------------------------------
   0.0    0.5   0.6585 0.0005 0.4823 0.8347   True
--------------------------------------------------

=== BST ===

Variável: casp9

Levene p-valor (casp9) = nan -> heterocedasticidade
Concentração 0.0: p = 0.6955 -> distribuição normal
Concentração 0.5: p = nan -> não normal

=== Welch ANOVA (heterocedasticidade) ===
          Source  ddof1     ddof2          F     p-unc       np2
0  concentration      1  1.3

# ROS

In [33]:
df = pd.read_excel("../../bancos de dados/dataset_ros_geral.xlsx")

df_BCT = df[df["type"] != "BST"]
df_BST = df[df["type"] != "BCT"]

df_BCT = df_BCT.copy()
df_BST = df_BST.copy()

bct_c = []

for i in range(len(df_BCT)):
    bct_c.append('BCT')

bst_c = []

for i in range(len(df_BST)):
    bst_c.append('BST')

df_BCT['type'] = bct_c
df_BST['type'] = bst_c

display(df_BCT)
display(df_BST)

,sample,type,concentration,intensity,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9
0,Snap-13,BCT,0.0,27157.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Snap-15,BCT,0.0,17222.0,NaN,NaN,NaN,NaN,NaN,NaN
2,Snap-17,BCT,0.0,19979.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Snap-19,BCT,0.0,NaN,NaN,220784.0,NaN,NaN,média,desvio padrão
4,Snap-22,BCT,0.0,22546.0,NaN,NaN,NaN,control,22125.375,6344.075795
5,Snap-24,BCT,0.0,32644.0,NaN,NaN,NaN,BCT 0.5,51093.666667,36678.923171
6,Snap-26,BCT,0.0,25226.0,NaN,NaN,NaN,BST 0.5,52405.111111,31005.159314
7,Snap-28,BCT,0.0,11927.0,NaN,NaN,NaN,BCT 1,66658.666667,49988.023795
8,Snap-30,BCT,0.0,20302.0,NaN,NaN,NaN,BST 1,95389.222222,65785.86666
9,Snap-181-Image Export-02.tif,BCT,0.5,20526.0,31828.0,NaN,NaN,BCT 2,65720.909091,47895.153962


,sample,type,concentration,intensity,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9
0,Snap-13,BST,0.0,27157.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Snap-15,BST,0.0,17222.0,NaN,NaN,NaN,NaN,NaN,NaN
2,Snap-17,BST,0.0,19979.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Snap-19,BST,0.0,NaN,NaN,220784.0,NaN,NaN,média,desvio padrão
4,Snap-22,BST,0.0,22546.0,NaN,NaN,NaN,control,22125.375,6344.075795
5,Snap-24,BST,0.0,32644.0,NaN,NaN,NaN,BCT 0.5,51093.666667,36678.923171
6,Snap-26,BST,0.0,25226.0,NaN,NaN,NaN,BST 0.5,52405.111111,31005.159314
7,Snap-28,BST,0.0,11927.0,NaN,NaN,NaN,BCT 1,66658.666667,49988.023795
8,Snap-30,BST,0.0,20302.0,NaN,NaN,NaN,BST 1,95389.222222,65785.86666
19,Snap-167-Image Export-16.tif,BST,0.5,56845.0,77172.0,NaN,NaN,BST 0.5,52405.111111,31005.159314


In [34]:
features = ['intensity']

compounds = {'BCT': df_BCT, 'BST': df_BST}

# rodar testes individuais por composto
for compound_name, df in compounds.items():
    print(f"\n=== {compound_name} ===")
    for feature in features:
        print(f"\nVariável: {feature}")
        print()

        # testes de pré-condição
        homoced = run_levene(df, feature)
        normal = run_shapiro(df, feature)

        # ANOVA adequada
        method = run_anova_oneway(df, feature, normal=normal, homoced=homoced)

        # Pós-teste adequado
        run_posthoc(df, feature, method=method)

# rodar two-way ANOVA com todos os compostos juntos
df_all = pd.concat([df_BCT.assign(compound='BCT'),
                    df_BST.assign(compound='BST')],
                   ignore_index=True)

for feature in features:
    print(f"\n=== ANOVA Two-way: {feature} ===")
    run_anova_twoway(df_all, feature)



=== BCT ===

Variável: intensity

Levene p-valor (intensity) = nan -> heterocedasticidade
Concentração 0.0: p = nan -> não normal
Concentração 0.5: p = nan -> não normal
Concentração 1.0: p = 0.0078 -> não normal
Concentração 2.0: p = 0.0219 -> não normal

=== Kruskal-Wallis (não paramétrico) ===
estatística = nan, p = nan

--- Dunn test (pós-hoc não paramétrico) ---
          0.0       0.5       1.0       2.0
0.0  1.000000  0.434616  0.016263  0.028389
0.5  0.434616  1.000000  1.000000  1.000000
1.0  0.016263  1.000000  1.000000  1.000000
2.0  0.028389  1.000000  1.000000  1.000000

=== BST ===

Variável: intensity

Levene p-valor (intensity) = nan -> heterocedasticidade
Concentração 0.0: p = nan -> não normal
Concentração 0.5: p = nan -> não normal
Concentração 1.0: p = 0.2202 -> distribuição normal
Concentração 2.0: p = 0.0042 -> não normal

=== Kruskal-Wallis (não paramétrico) ===
estatística = nan, p = nan

--- Dunn test (pós-hoc não paramétrico) ---
          0.0       0.5      